workbook for migrating database: filemaker to sqlite

In [7]:
import sqlite3, os
import pandas as pd
import re

In [9]:
#set home directory path
hdir = os.path.expanduser('~')

dh_path = '/Dropbox/Active_Directories/Digital_Humanities/Persian_Dictionary/'

database_path = os.path.join(hdir, dh_path.strip('/'), 'Persian_Pahlavi_Dictionary.db')

In [11]:

# Check if database file exists
if not os.path.exists(database_path):
    raise FileNotFoundError(f"Database file not found at: {database_path}")

# Connect to the SQLite database
conn = sqlite3.connect(database_path)

# Create a cursor object to execute SQL commands
cursor = conn.cursor()

try:
    # Your database operations will go here
    pass
finally:
    # Always close the connection when done
    cursor.close()
    conn.close()

In [13]:
# Connect to the SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

try:
    # Query to get all table names
    cursor.execute("""
        SELECT name 
        FROM sqlite_master 
        WHERE type='table'
        ORDER BY name;
    """)
    
    # Fetch all results
    tables = cursor.fetchall()
    
    print("Tables in database:")
    for table in tables:
        table_name = table[0]
        # Query to get the number of columns and foreign keys for each table
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns_info = cursor.fetchall()
        num_columns = len(columns_info)

        cursor.execute(f"PRAGMA foreign_key_list({table_name});")
        foreign_keys = cursor.fetchall()
        foreign_keys_info = [fk[3] for fk in foreign_keys]  # Get the foreign key names

        print(f"- Table: {table_name}, Number of Columns: {num_columns}, Foreign Keys: {foreign_keys_info}")

finally:
    cursor.close()
    conn.close()

Tables in database:
- Table: definitions_pahlavi, Number of Columns: 6, Foreign Keys: ['Word_ID', 'Scholarship_ID']
- Table: dehkhoda_dictionary, Number of Columns: 3, Foreign Keys: []
- Table: examples_pahlavi, Number of Columns: 8, Foreign Keys: ['Word_ID', 'Source_ID', 'Scholarship_ID']
- Table: malandra_pahlavi_dictionary, Number of Columns: 29, Foreign Keys: []
- Table: related_words_pahlavi, Number of Columns: 5, Foreign Keys: ['Parent_ID', 'Child_ID']
- Table: scholarship, Number of Columns: 3, Foreign Keys: []
- Table: sources_pahlavi, Number of Columns: 4, Foreign Keys: []
- Table: steingass_dictionary, Number of Columns: 9, Foreign Keys: []
- Table: vonmelzer_dictionary, Number of Columns: 10, Foreign Keys: []
- Table: words_pahlavi, Number of Columns: 11, Foreign Keys: []


In [18]:
def get_unique_values(table_name, column_name):
    """
    Retrieve a list of all unique values in the specified column of a table.
    """
    # Establish a connection to the database using the database_path
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    
    # Query to select distinct values from the specified column
    query = f"SELECT DISTINCT {column_name} FROM {table_name};"
    cursor.execute(query)
    
    # Fetch all unique values
    unique_values = [row[0] for row in cursor.fetchall()]
    
    # Close the cursor and connection
    cursor.close()
    conn.close()
    
    return unique_values

In [20]:
def replace_linebreaks_with_comma(table_name, column_name):
    """
    Replace line break delimiters in a specified column with a comma.
    """
    # Establish a connection to the database
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    
    # Define the line break delimiters to replace
    linebreaks = ['\x0b', '\n', '\r']
    
    # Create a SQL query to update the column
    for linebreak in linebreaks:
        query = f"""
        UPDATE {table_name}
        SET {column_name} = REPLACE({column_name}, ?, ?)
        """
        cursor.execute(query, (linebreak, ','))
    
    # Commit the changes and close the connection
    conn.commit()
    cursor.close()
    conn.close()

In [5]:
#replace_linebreaks_with_comma("bibliography", "Language")

In [6]:
#get_unique_values("bibliography", "Language")

In [29]:
# Function to enable regex in SQLite
def _regex_search(pattern, string):
    # Check if the string is valid
    if not isinstance(string, str):
        return False
    try:
        return re.search(pattern, string) is not None
    except Exception as e:
        print(f"Regex error: {e}")
        return False

# Register the regex function with SQLite
def _register_regex(conn):
    conn.create_function("REGEXP", 2, _regex_search)

In [43]:
def simp_pers_search(search_term, max_results=5):
    """
    Search across multiple Persian dictionary tables for a given term.
    
    Args:
        search_term (str): The Persian term to search for (supports regex)
        max_results (int): Maximum number of results to display per dictionary (default: 5)
    
    Returns:
        None (prints formatted results)
    """
    conn = sqlite3.connect(database_path)
    _register_regex(conn)
    cursor = conn.cursor()
    
    print(f"🔍 Searching for: '{search_term}' (showing up to {max_results} results per dictionary)")
    print("=" * 80)
    
    try:
        # 1. Search Dehkhoda Dictionary
        # First get total count
        cursor.execute("""
            SELECT COUNT(*) 
            FROM dehkhoda_dictionary 
            WHERE Word REGEXP ?
        """, (search_term,))
        dehkhoda_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT Word, Definition 
            FROM dehkhoda_dictionary 
            WHERE Word REGEXP ?
            ORDER BY LENGTH(Word)
            LIMIT ?
        """, (search_term, max_results))
        
        dehkhoda_results = cursor.fetchall()
        print(f"📚 DEHKHODA DICTIONARY TABLE (displaying {len(dehkhoda_results)} out of {dehkhoda_total} matches)")
        print("-" * 40)
        if dehkhoda_results:
            for i, (word, definition) in enumerate(dehkhoda_results, 1):
                print(f"{i}. {word}")
                print(f"   📖 {definition}")
                print()
        else:
            print("   No matches found\n")
        
        # 2. Search Steingass Dictionary
        # First get total count
        cursor.execute("""
            SELECT COUNT(*) 
            FROM steingass_dictionary 
            WHERE headword_persian REGEXP ?
        """, (search_term,))
        steingass_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT headword_persian, definitions 
            FROM steingass_dictionary 
            WHERE headword_persian REGEXP ?
            ORDER BY LENGTH(headword_persian)
            LIMIT ?
        """, (search_term, max_results))
        
        steingass_results = cursor.fetchall()
        print(f"📚 STEINGASS DICTIONARY TABLE (displaying {len(steingass_results)} out of {steingass_total} matches)")
        print("-" * 40)
        if steingass_results:
            for i, (headword, definition) in enumerate(steingass_results, 1):
                print(f"{i}. {headword}")
                print(f"   📖 {definition}")
                print()
        else:
            print("   No matches found\n")
        
        # 3. Search Von Melzer Dictionary
        # First get total count
        cursor.execute("""
            SELECT COUNT(*) 
            FROM vonmelzer_dictionary 
            WHERE `Präs.-Stamm` REGEXP ?
        """, (search_term,))
        vonmelzer_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT `Präs.-Stamm`, English_Def 
            FROM vonmelzer_dictionary 
            WHERE `Präs.-Stamm` REGEXP ?
            ORDER BY LENGTH(`Präs.-Stamm`)
            LIMIT ?
        """, (search_term, max_results))
        
        vonmelzer_results = cursor.fetchall()
        print(f"📚 VON MELZER DICTIONARY TABLE (displaying {len(vonmelzer_results)} out of {vonmelzer_total} matches)")
        print("-" * 40)
        if vonmelzer_results:
            for i, (pres_stamm, english_def) in enumerate(vonmelzer_results, 1):
                print(f"{i}. {pres_stamm}")
                print(f"   📖 {english_def}")
                print()
        else:
            print("   No matches found\n")
        
        # 4. Search Words Pahlavi with Definitions (most complex)
        # First get total count
        cursor.execute("""
            SELECT COUNT(*) 
            FROM words_pahlavi 
            WHERE New_Persian REGEXP ?
        """, (search_term,))
        pahlavi_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT 
                wp.New_Persian,
                wp.Transcription,
                wp.Translit_Skj,
                wp.Translit_Mac,
                dp.Definition
            FROM words_pahlavi wp
            LEFT JOIN definitions_pahlavi dp ON wp.UID = dp.Word_ID
            WHERE wp.New_Persian REGEXP ?
            ORDER BY LENGTH(wp.New_Persian)
            LIMIT ?
        """, (search_term, max_results))
        
        pahlavi_results = cursor.fetchall()
        print(f"📚 PAHLAVI WORDS & DEFINITIONS TABLE (displaying {len(pahlavi_results)} out of {pahlavi_total} matches)")
        print("-" * 40)
        if pahlavi_results:
            # Group results by word to handle multiple definitions
            word_groups = {}
            for new_persian, transcription, translit_skj, translit_mac, definition in pahlavi_results:
                key = (new_persian, transcription, translit_skj, translit_mac)
                if key not in word_groups:
                    word_groups[key] = []
                if definition:  # Only add non-null definitions
                    word_groups[key].append(definition)
            
            for i, ((new_persian, transcription, translit_skj, translit_mac), definitions) in enumerate(word_groups.items(), 1):
                print(f"{i}. {new_persian}")
                if transcription:
                    print(f"   🔤 Transcription: {transcription}")
                
                # Use Translit_Skj if available, otherwise Translit_Mac, otherwise skip
                transliteration = translit_skj if translit_skj else translit_mac
                if transliteration:
                    print(f"   🔤 Transliteration: {transliteration}")
                
                if definitions:
                    print(f"   📖 Definitions:")
                    for j, definition in enumerate(definitions, 1):
                        print(f"      {j}. {definition}")
                else:
                    print(f"   📖 No definitions available")
                print()
        else:
            print("   No matches found\n")
        
        # Summary
        total_results = len(dehkhoda_results) + len(steingass_results) + len(vonmelzer_results) + len(pahlavi_results)
        print("=" * 80)
        print(f"📊 SUMMARY: {total_results} results displayed (limited to {max_results} per dictionary)")
        print(f"   Dehkhoda: {len(dehkhoda_results)} | Steingass: {len(steingass_results)} | Von Melzer: {len(vonmelzer_results)} | Pahlavi: {len(pahlavi_results)}")
        
    except Exception as e:
        print(f"❌ Search error: {e}")
    finally:
        cursor.close()
        conn.close()


# Example usage function for testing
def test_search():
    """Test the search function with sample terms"""
    print("Testing search function with default limit (5 results)...")
    simp_pers_search("خانه")  # Search for "house" in Persian
    
    print("\n" + "="*80 + "\n")
    
    print("Testing search function with increased limit (10 results)...")
    simp_pers_search(".*ان.*", max_results=10)  # Search for words containing "ان"

In [45]:
    simp_pers_search("خانه")  # Search for "house" in Persian

🔍 Searching for: 'خانه' (showing up to 5 results per dictionary)
📚 DEHKHODA DICTIONARY TABLE (displaying 5 out of 73 matches)
----------------------------------------
1. خانه
   📖 1 سرا، منزل، مسکن، دار، بیت 2 آپارتمان، کاشانه 3 اتاق، وثاق، حجره 4 چاردیواری، سرپناه 5 اقامتگاه، ساختمان‌مسکونی، ماوا 6 کلبه، سراچه 7 آلونک، کوخ 8 دولت‌سرا، قصر، کاخ 9 لانه، آشیانه 01 غار، سوراخ، کنام 11 بوم 21 میهن،

2. غمخانه
   📖 غم‌آباد، غمستان، غمسرا، غمکده، ماتم‌سرا، محنتستان، محنت‌سرا

3. چارخانه
   📖 شطرنجی

4. خانه‌پا
   📖 سرایدار

5. دیوخانه
   📖 دیوزار، دیوستان، دیوکده

📚 STEINGASS DICTIONARY TABLE (displaying 5 out of 342 matches)
----------------------------------------
1. خانه
   📖 A house, dwelling, habitation; a tent, pavilion; a receptacle; a drawer, partition, compartment; department; the arm from the shoulder to the elbow; a field; a woman; a page of letter-paper; a heap of corn; a hillock of sand; verse, poetry; — *ḵẖāna’i āfat-rez*, The world; — *ḵẖāna’i āwang-angūr*, A place for suspend

In [67]:
def pah_simp_search(search_term, max_results=5):
    """
    Search for Pahlavi terms across multiple fields and related tables.
    
    Args:
        search_term (str): The Pahlavi term to search for (supports regex)
        max_results (int): Maximum number of results to display per section (default: 5)
    
    Returns:
        None (prints formatted results)
    """
    conn = sqlite3.connect(database_path)
    _register_regex(conn)
    cursor = conn.cursor()
    
    print(f"🔍 Pahlavi Search for: '{search_term}' (showing up to {max_results} results per section)")
    print("=" * 80)
    
    try:
        # 1. Search Pahlavi words in Transcription, Translit_Mac, and Translit_Skj
        cursor.execute("""
            SELECT COUNT(DISTINCT wp.UID)
            FROM words_pahlavi wp
            WHERE wp.Transcription REGEXP ? 
               OR wp.Translit_Mac REGEXP ? 
               OR wp.Translit_Skj REGEXP ?
        """, (search_term, search_term, search_term))
        pahlavi_words_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT DISTINCT
                wp.UID,
                wp.Transcription,
                wp.Translit_Mac,
                wp.Translit_Skj,
                wp.New_Persian,
                dp.Type,
                dp.Definition
            FROM words_pahlavi wp
            LEFT JOIN definitions_pahlavi dp ON wp.UID = dp.Word_ID
            WHERE wp.Transcription REGEXP ? 
               OR wp.Translit_Mac REGEXP ? 
               OR wp.Translit_Skj REGEXP ?
            ORDER BY LENGTH(COALESCE(wp.Transcription, wp.Translit_Mac, wp.Translit_Skj))
            LIMIT ?
        """, (search_term, search_term, search_term, max_results))
        
        pahlavi_words_results = cursor.fetchall()
        matched_uids = [row[0] for row in pahlavi_words_results]  # Store UIDs for later use
        new_persian_values = [row[4] for row in pahlavi_words_results if row[4]]  # Store New Persian values
        
        print(f"📚 PAHLAVI WORDS & DEFINITIONS (displaying {len(pahlavi_words_results)} out of {pahlavi_words_total} matches)")
        print("-" * 40)
        if pahlavi_words_results:
            for i, (uid, transcription, translit_mac, translit_skj, new_persian, def_type, definition) in enumerate(pahlavi_words_results, 1):
                # Show the matched transcription/transliteration
                matched_form = transcription or translit_mac or translit_skj
                print(f"{i}. {matched_form}")
                if new_persian:
                    print(f"   🔤 New Persian: {new_persian}")
                
                if def_type and definition:
                    print(f"   📖 {def_type}: {definition}")
                elif definition:
                    print(f"   📖 {definition}")
                else:
                    print(f"   📖 No definition available")
                print()
        else:
            print("   No matches found\n")
        
        # 2. Search Examples using matched UIDs
        if matched_uids:
            cursor.execute(f"""
                SELECT COUNT(*)
                FROM examples_pahlavi 
                WHERE Word_ID IN ({','.join(['?' for _ in matched_uids])})
            """, matched_uids)
            examples_total = cursor.fetchone()[0]
            
            cursor.execute(f"""
                SELECT 
                    ep.Example,
                    ep.Translation,
                    ep.Citation,
                    sp.Title
                FROM examples_pahlavi ep
                LEFT JOIN sources_pahlavi sp ON ep.Source_ID = sp.UID
                WHERE ep.Word_ID IN ({','.join(['?' for _ in matched_uids])})
                ORDER BY LENGTH(ep.Example)
                LIMIT ?
            """, matched_uids + [max_results])
            
            examples_results = cursor.fetchall()
            print(f"📚 PAHLAVI EXAMPLES (displaying {len(examples_results)} out of {examples_total} matches)")
            print("-" * 40)
            if examples_results:
                for i, (example, translation, citation, source_title) in enumerate(examples_results, 1):
                    if example:
                        print(f"{i}. {example}")
                    if translation:
                        print(f"   🔄 {translation}")
                    if source_title:
                        print(f"   📚 Source: {source_title}")
                    if citation:
                        print(f"   📝 Citation: {citation}")
                    print()
            else:
                print("   No examples found\n")
        
        # 3. Search Malandra Pahlavi Dictionary
        cursor.execute("""
            SELECT COUNT(*)
            FROM malandra_pahlavi_dictionary 
            WHERE transcription_skj REGEXP ?
        """, (search_term,))
        malandra_total = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT transcription_skj, part_of_speech, definition, etymology
            FROM malandra_pahlavi_dictionary 
            WHERE transcription_skj REGEXP ?
            ORDER BY LENGTH(transcription_skj)
            LIMIT ?
        """, (search_term, max_results))
        
        malandra_results = cursor.fetchall()
        print(f"📚 MALANDRA PAHLAVI DICTIONARY (displaying {len(malandra_results)} out of {malandra_total} matches)")
        print("-" * 40)
        if malandra_results:
            for i, (transcription_skj, part_of_speech, definition, etymology) in enumerate(malandra_results, 1):
                print(f"{i}. {transcription_skj}")
                if part_of_speech:
                    print(f"   📝 Part of Speech: {part_of_speech}")
                if definition:
                    print(f"   📖 Definition: {definition}")
                if etymology:
                    print(f"   🌱 Etymology: {etymology}")
                print()
        else:
            print("   No matches found\n")
        
        # 4. New Persian lookup section
        if new_persian_values:
            print("🔗 NEW PERSIAN CONNECTIONS")
            print("=" * 80)
            
            unique_new_persian = list(set(new_persian_values))  # Remove duplicates
            
            for np_value in unique_new_persian[:max_results]:  # Limit New Persian lookups
                print(f"🔍 Looking up New Persian: '{np_value}'")
                print("-" * 50)
                
                # Search Dehkhoda - get total count first
                cursor.execute("""
                    SELECT COUNT(*) 
                    FROM dehkhoda_dictionary 
                    WHERE Word = ?
                """, (np_value,))
                dehkhoda_np_total = cursor.fetchone()[0]
                
                cursor.execute("""
                    SELECT Word, Definition 
                    FROM dehkhoda_dictionary 
                    WHERE Word = ?
                    LIMIT ?
                """, (np_value, max_results))
                
                dehkhoda_np_results = cursor.fetchall()
                if dehkhoda_np_results:
                    print(f"📚 Dehkhoda Dictionary (displaying {len(dehkhoda_np_results)} out of {dehkhoda_np_total} matches):")
                    for word, definition in dehkhoda_np_results:
                        print(f"   📖 {definition}")
                
                # Search Steingass - get total count first
                cursor.execute("""
                    SELECT COUNT(*) 
                    FROM steingass_dictionary 
                    WHERE headword_persian = ?
                """, (np_value,))
                steingass_np_total = cursor.fetchone()[0]
                
                cursor.execute("""
                    SELECT headword_persian, definitions 
                    FROM steingass_dictionary 
                    WHERE headword_persian = ?
                    LIMIT ?
                """, (np_value, max_results))
                
                steingass_np_results = cursor.fetchall()
                if steingass_np_results:
                    print(f"📚 Steingass Dictionary (displaying {len(steingass_np_results)} out of {steingass_np_total} matches):")
                    for headword, definition in steingass_np_results:
                        print(f"   📖 {definition}")
                
                # Search Von Melzer - get total count first
                cursor.execute("""
                    SELECT COUNT(*) 
                    FROM vonmelzer_dictionary 
                    WHERE `Präs.-Stamm` = ?
                """, (np_value,))
                vonmelzer_np_total = cursor.fetchone()[0]
                
                cursor.execute("""
                    SELECT `Präs.-Stamm`, English_Def 
                    FROM vonmelzer_dictionary 
                    WHERE `Präs.-Stamm` = ?
                    LIMIT ?
                """, (np_value, max_results))
                
                vonmelzer_np_results = cursor.fetchall()
                if vonmelzer_np_results:
                    print(f"📚 Von Melzer Dictionary (displaying {len(vonmelzer_np_results)} out of {vonmelzer_np_total} matches):")
                    for pres_stamm, english_def in vonmelzer_np_results:
                        print(f"   📖 {english_def}")
                
                if not (dehkhoda_np_results or steingass_np_results or vonmelzer_np_results):
                    print("   No exact matches found in Persian dictionaries")
                print()
        
        # Summary
        total_pahlavi = len(pahlavi_words_results) + len(examples_results if matched_uids else []) + len(malandra_results)
        total_new_persian = len(unique_new_persian) if new_persian_values else 0
        
        print("=" * 80)
        print(f"📊 SUMMARY: {total_pahlavi} Pahlavi results, {total_new_persian} New Persian lookups")
        print(f"   Pahlavi Words: {len(pahlavi_words_results)} | Examples: {len(examples_results if matched_uids else [])} | Malandra: {len(malandra_results)}")
        
    except Exception as e:
        print(f"❌ Search error: {e}")
    finally:
        cursor.close()
        conn.close()


# Example usage function for testing
def test_pahlavi_search():
    """Test the Pahlavi search function with sample terms"""
    print("Testing Pahlavi search function with default limit (5 results)...")
    pah_simp_search("xwad")  # Search for Pahlavi "xwad" (self)
    
    print("\n" + "="*80 + "\n")
    
    print("Testing Pahlavi search function with increased limit (8 results)...")
    pah_simp_search(".*an.*", max_results=8)  # Search for Pahlavi terms containing "an"

In [69]:
pah_simp_search("xāna")


🔍 Pahlavi Search for: 'xāna' (showing up to 5 results per section)
📚 PAHLAVI WORDS & DEFINITIONS (displaying 2 out of 1 matches)
----------------------------------------
1. xānag
   🔤 New Persian: خانه
   📖 definition: house

2. xānag
   🔤 New Persian: خانه
   📖 definition: room; quite common for خانه to mean 'room' in many dialects, and sometimes seems to have that meaning in Pahlavi as well

📚 PAHLAVI EXAMPLES (displaying 0 out of 0 matches)
----------------------------------------
   No examples found

📚 MALANDRA PAHLAVI DICTIONARY (displaying 1 out of 1 matches)
----------------------------------------
1. xānag
   📖 Definition: 'house'
   🌱 Etymology: Aram byt>; NPers xānah.

🔗 NEW PERSIAN CONNECTIONS
🔍 Looking up New Persian: 'خانه'
--------------------------------------------------
📚 Dehkhoda Dictionary (displaying 1 out of 1 matches):
   📖 1 سرا، منزل، مسکن، دار، بیت 2 آپارتمان، کاشانه 3 اتاق، وثاق، حجره 4 چاردیواری، سرپناه 5 اقامتگاه، ساختمان‌مسکونی، ماوا 6 کلبه، سراچه 7 آلونک،

In [65]:

def word_search(search_term):
    # Establish a connection to the database
    conn = sqlite3.connect(database_path)
    _register_regex(conn)  # Register the regex function
    cursor = conn.cursor()
    
    try:
        # SQL query to search through multiple columns in the lexicon table and join on definitions
        query = """
        SELECT l.UID, l.Term, l.Translation, l.Emic_Term, l.Colonial_Term, l.Transliteration, d.Definition
        FROM lexicon l
        JOIN definitions d ON l.UID = d.Lexicon_ID
        WHERE l.Term REGEXP ? OR l.Translation REGEXP ? OR l.Emic_Term REGEXP ? OR l.Colonial_Term REGEXP ? OR l.Transliteration REGEXP ?;
        """
        
        # Execute the query with the search term
        cursor.execute(query, (search_term, search_term, search_term, search_term, search_term))
        
        # Fetch all results
        results = cursor.fetchall()
        
        # Create a DataFrame with labeled columns
        columns = ['UID', 'Term', 'Translation', 'Emic_Term', 'Colonial_Term', 'Transliteration', 'Definition']
        df = pd.DataFrame(results, columns=columns)
        
        # Set display options for long text
        pd.set_option('display.max_colwidth', None)  # Show full content of the Definition column
        
        # Initialize Related_Terms column
        df['Related_Terms'] = ''
        
        # Check for related terms
        for index, row in df.iterrows():
            uid = row['UID']
            # Query to find related terms
            related_query = """
            SELECT l.Term
            FROM related_terms rt
            JOIN lexicon l ON rt.Child_ID = l.UID
            WHERE rt.Parent_ID = ?;
            """
            cursor.execute(related_query, (uid,))
            related_terms = cursor.fetchall()
            # Concatenate related terms into a single string
            related_terms_list = [term[0] for term in related_terms]
            df.at[index, 'Related_Terms'] = ', '.join(related_terms_list)
        
    except Exception as e:
        print(f"Error during query execution: {e}")
    finally:
        # Close the cursor and connection
        cursor.close()
        conn.close()
    
    return df

# Example usage
# search_results_df = word_search('your_regex_pattern')
# print(search_results_df)

In [25]:
word_search ('باج')

# next step: optional second argument that filters based on tags

NameError: name 'regex_search' is not defined

In [13]:
# Establish a connection to the database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

try:
    # Update the UID column to remove leading quotation marks
    cursor.execute("""
        UPDATE words_pahlavi
        SET UID = REPLACE(UID, '"', '')
        WHERE UID LIKE '"%';
    """)

    # Commit the changes
    conn.commit()

finally:
    # Close the cursor and connection
    cursor.close()
    conn.close()

In [20]:
# Establish a connection to the database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

try:
    # Check for missing Word_IDs in definitions_pahlavi that do not exist in words_pahlavi
    cursor.execute("""
        SELECT DISTINCT d.Word_ID
        FROM definitions_pahlavi d
        LEFT JOIN words_pahlavi w ON d.Word_ID = w.UID
        WHERE w.UID IS NULL AND d.Word_ID IS NOT NULL;
    """)
    missing_word_ids = cursor.fetchall()

    # Check for NULL values in the Word_ID column
    cursor.execute("""
        SELECT COUNT(*) 
        FROM definitions_pahlavi 
        WHERE Word_ID IS NULL;
    """)
    null_word_ids_count = cursor.fetchone()[0]

    # Output the results
    print("Missing Word_IDs in definitions_pahlavi that do not exist in words_pahlavi:")
    for word_id in missing_word_ids:
        print(word_id[0])  # Print each missing Word_ID

    print(f"\nNumber of NULL values in Word_ID column: {null_word_ids_count}")

finally:
    # Close the cursor and connection
    cursor.close()
    conn.close()

Missing Word_IDs in definitions_pahlavi that do not exist in words_pahlavi:

Number of NULL values in Word_ID column: 0


In [36]:
# Code to auto-translate German definitions into English


# from googletrans import Translator
# import sqlite3
# import time

# # Establish a connection to the database
# conn = sqlite3.connect(database_path)
# cursor = conn.cursor()

# try:
#     # Initialize the translator
#     translator = Translator()

#     # Set the starting point and chunk size
#     start_row = 1001  # Starting from row 1001
#     chunk_size = 500

#     while True:
#         # Fetch the next chunk of rows
#         cursor.execute(f"SELECT rowid, Deutsch FROM \"von-melzer_dictionary\" LIMIT {chunk_size} OFFSET {start_row - 1};")
#         rows = cursor.fetchall()

#         # Break the loop if there are no more rows to process
#         if not rows:
#             print("No more rows to process.")
#             break

#         # Update the English_Def column with translations
#         for row in rows:
#             rowid, deutsch_value = row
#             if deutsch_value:  # Check if the value is not None
#                 try:
#                     translation = translator.translate(deutsch_value, src='de', dest='en').text
#                     cursor.execute("UPDATE \"von-melzer_dictionary\" SET English_Def = ? WHERE rowid = ?;", (translation, rowid))
#                     print(f"Translated: {deutsch_value} -> {translation}")  # Print the translation for verification
#                 except Exception as e:
#                     print(f"Error translating '{deutsch_value}': {e}")
#                     time.sleep(1)  # Wait a bit before retrying

#         # Commit the changes after processing the chunk
#         conn.commit()

#         # Update the starting point for the next chunk
#         start_row += chunk_size

# finally:
#     cursor.close()
#     conn.close()

## Search Functions

### Pahlavi Transcription Search

In [117]:
import sqlite3
import re

def regex_search(pattern, string):
    """Check if the string matches the regex pattern."""
    if not isinstance(string, str):
        return False
    try:
        return re.search(pattern, string) is not None
    except re.error as e:
        print(f"Regex error: {e}")
        return False

def pah_transcript_search(search_term):
    """
    Search for terms in the Transcription column of the words_pahlavi table
    that match the provided regex search term.
    
    Args:
        search_term (str): The regex search term to use for matching.
        
    Returns:
        str: A report of the matched terms and their definitions.
    """
    # Establish a connection to the database
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    
    try:
        # Register the regex function with SQLite
        conn.create_function("REGEXP", 2, regex_search)

        # SQL query to search for general matches in the Transcription column
        query = """
        SELECT Transcription, UID, New_Persian
        FROM words_pahlavi
        WHERE Transcription REGEXP ?;
        """
        
        # Execute the query with the search term
        cursor.execute(query, (search_term,))
        
        # Fetch all matching records
        matching_terms = cursor.fetchall()
        
        # Extract terms, UIDs, and New_Persian from the results
        matched_terms_list = [(term[0], term[1], term[2]) for term in matching_terms]
        total_matches = len(matched_terms_list)

        # Prepare the output message
        if total_matches > 0:
            # Find exact matches (for stricter matching, we can use anchors)
            exact_matches = [(term, uid, new_persian) for term, uid, new_persian in matched_terms_list if re.fullmatch(search_term, term)]
            exact_matches_count = len(exact_matches)

            # Prepare the list of general matches (first 10)
            general_matches = matched_terms_list[:10]
            remaining_matches_count = total_matches - len(general_matches)

            # Construct the report
            report = (
                f"Your search ({search_term}) matched {total_matches} entries.\n"
                f"Exact matches: {', '.join(term for term, _, _ in exact_matches)}\n"
                f"All matches: {', '.join(term for term, _, _ in general_matches)}"
            )
            if remaining_matches_count > 0:
                report += f" ... {remaining_matches_count} more matches."

            # Two line breaks before definitions
            report += "\n\n"

            # Look up definitions for exact matches
            if exact_matches:
                for term, uid, new_persian in exact_matches:
                    # Query for definitions corresponding to the exact match
                    definition_query = """
                    SELECT Definition
                    FROM definitions_pahlavi
                    WHERE Word_ID = ?;
                    """
                    cursor.execute(definition_query, (uid,))
                    definitions = cursor.fetchall()

                    # Prepare definitions output
                    definitions_output = [definition[0] for definition in definitions]
                    report += f"Definitions for {term}:\n"
                    for definition in definitions_output:
                        report += f"    {definition}\n"  # Indentation for definitions

                    # Check for New Persian entry and look it up in the dehkhoda_dictionary
                    if new_persian:
                        # Query the dehkhoda_dictionary
                        dehkhoda_query = """
                        SELECT Definition
                        FROM dehkhoda_dictionary
                        WHERE Word = ?;
                        """
                        cursor.execute(dehkhoda_query, (new_persian,))
                        dehkhoda_definitions = cursor.fetchall()

                        # Add Dehkhoda definitions to the report
                        if dehkhoda_definitions:
                            report += f"    Definitions from Dehkhoda for {new_persian}:\n"
                            for dehkhoda_definition in dehkhoda_definitions:
                                report += f"        {dehkhoda_definition[0]}\n"  # Indentation for Dehkhoda definitions

                        # Query the steingass_dictionary
                        steingass_query = """
                        SELECT definitions
                        FROM steingass_dictionary
                        WHERE headword_persian = ?;
                        """
                        cursor.execute(steingass_query, (new_persian,))
                        steingass_definitions = cursor.fetchall()

                        # Add Steingass definitions to the report
                        if steingass_definitions:
                            report += f"    Definitions from Steingass for {new_persian}:\n"
                            for steingass_definition in steingass_definitions:
                                report += f"        {steingass_definition[0]}\n"  # Indentation for Steingass definitions

                    # Look for definitions in the malandra_pahlavi_dictionary
                    malandra_query = """
                    SELECT definition
                    FROM malandra_pahlavi_dictionary
                    WHERE transcription_skj = ?;
                    """
                    cursor.execute(malandra_query, (term,))
                    malandra_definitions = cursor.fetchall()

                    # Add Malandra definitions to the report
                    if malandra_definitions:
                        report += f"    Definitions from Malandra for {term}:\n"
                        for malandra_definition in malandra_definitions:
                            report += f"        {malandra_definition[0]}\n"  # Indentation for Malandra definitions

        else:
            report = f"No terms matched your regex search ({search_term})."
        
        return report
    
    except Exception as e:
        return f"An error occurred: {e}"
    
    finally:
        # Close the cursor and connection
        cursor.close()
        conn.close()

# Example usage
# print(pah_transcript_search('your_regex_pattern'))

In [118]:
print(pah_transcript_search('dōst'))

Your search (dōst) matched 3 entries.
Exact matches: dōst, dōst
All matches: dōst, dōstīh, dōst

Definitions for dōst:
    friend.
    Definitions from Dehkhoda for دوست:
        آشنا، حبیب، خلیل، دمساز، دوستدار، رفیق، صحابه، صدوق، صدیق، محب، محبوب، محبوبه، مصاحب، معاشر، معشوق، ولی، همراه، همنشین، یار & دشمن، عدو
    Definitions from Steingass for دوست:
        A friend; a lover; a mistress, sweetheart; (for دو است *du ast*) are two; — *dost dāshtan*, To like, love, be fond of, esteem as a friend.
    Definitions from Malandra for dōst:
        n. 'friend'
Definitions for dōst:
    friend  04
    Definitions from Malandra for dōst:
        n. 'friend'



In [183]:
import sqlite3
import re

def regex_search(pattern, string):
    """Check if the string matches the regex pattern."""
    if not isinstance(string, str):
        return False
    try:
        return re.search(pattern, string) is not None
    except re.error as e:
        print(f"Regex error: {e}")
        return False

def pah_uid_report(uid):
    # (Assuming the implementation of pah_uid_report is already defined here)
    # This function should return the formatted report for the given UID.
    pass

def pah_uid_search(search_term):
    """
    Search for the regex term in the Transcription, Translit_Mac, and Translit_Skj columns
    of the words_pahlavi table, and in the transcription_skj column of the malandra_pahlavi_dictionary table.
    
    Args:
        search_term (str): The regex search term to use for matching.
        
    Returns:
        str: A report of the matched terms and their UIDs.
    """
    # Establish a connection to the database
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    
    try:
        # Register the regex function with SQLite
        conn.create_function("REGEXP", 2, regex_search)

        # Construct the regex pattern for stricter matching
        regex_pattern = f"^{search_term}$"

        # SQL query to search for exact matches in the words_pahlavi table
        query_words = """
        SELECT UID, Transcription
        FROM words_pahlavi
        WHERE Transcription REGEXP ? OR Translit_Mac REGEXP ? OR Translit_Skj REGEXP ?;
        """
        
        # Execute the query with the regex pattern
        cursor.execute(query_words, (regex_pattern, regex_pattern, regex_pattern))
        words_matches = cursor.fetchall()

        # SQL query to search for exact matches in the malandra_pahlavi_dictionary table
        query_malandra = """
        SELECT UID, transcription_skj
        FROM malandra_pahlavi_dictionary
        WHERE transcription_skj REGEXP ?;
        """
        
        # Execute the query with the regex pattern
        cursor.execute(query_malandra, (regex_pattern,))
        malandra_matches = cursor.fetchall()

        # Prepare the output message
        report = f"Exact matches for {search_term}:\n"
        
        # Format the exact matches from words_pahlavi
        if words_matches:
            for uid, transcription in words_matches:
                report += f"    {uid}: {transcription}\n"  # Indented
        else:
            report += "    None\n"

        # Add a line break before Malandra matches
        report += "\n    Malandra matches:\n"
        if malandra_matches:
            for uid, transcription_skj in malandra_matches:
                report += f"        {uid}: {transcription_skj}\n"  # Further indented
        else:
            report += "        None\n"

        # Add a line break before other possible matches
        report += "\nOther possible matches:\n"

        # SQL query for less restrictive matches (not exact matches)
        query_general = """
        SELECT UID, Transcription
        FROM words_pahlavi
        WHERE Transcription REGEXP ? OR Translit_Mac REGEXP ? OR Translit_Skj REGEXP ?;
        """
        
        # Execute the query with the regex pattern for general matches
        cursor.execute(query_general, (search_term, search_term, search_term))
        general_matches = cursor.fetchall()

        # Prepare general matches output
        general_matches_output = []
        if general_matches:
            for uid, transcription in general_matches:
                # Exclude exact matches from the general matches list
                if not any(uid == exact[0] for exact in words_matches):
                    general_matches_output.append(f"{uid}: {transcription}")

        # Format the general matches output
        if general_matches_output:
            if len(general_matches_output) > 30:
                report += "; ".join(general_matches_output[:30]) + f" ... [{len(general_matches_output) - 30} remaining matches]\n"
            else:
                report += "; ".join(general_matches_output) + "\n"
        else:
            report += "None\n"

        # Check for a single exact match and add additional report if applicable
        if len(words_matches) == 1:
            uid, transcription = words_matches[0]
            report += "\n\n\nReport on " + transcription + "\n\n"
            report += pah_uid_report(uid)  # Call pah_uid_report for the UID of the match

        return report
    
    except Exception as e:
        return f"An error occurred: {e}"
    
    finally:
        # Close the cursor and connection
        cursor.close()
        conn.close()

# Example usage
# print(pah_uid_search('ab.g'))

In [198]:
print(pah_uid_search('lšn'))

Exact matches for lšn:
    None

    Malandra matches:
        None

Other possible matches:
490: ārešn; 598: āwarišn; 625: awizīrišnīg; 745: bāliš; 773: baršnūm; 1010: cihršnāsīh; 1325: dwārišn; 1453: frašn; 2516: muhrišn; 2813: padnigerišn; 2912: parwarišn; 3148: Rašn; 3797: uzwān; 3890: warišn; 4076: wišāddwārišnīh; 4249: xwarišn; 4250: xwārišn; 4251: xwarišnīg; 4547: dārišn; 4778: abāz-dārišnīh; 4967: axwarišn; 5009: bē-barišnīh; 5012: bē-widerišnīh; 5155: dwārišnīh; 5549: nigerišn < niger-; 5691: Rašn-dād; 5882: wārišn; 5934: winnārišn; 5941: wišād-dwārišnīh; 5966: Wourubaršn ... [1 remaining matches]



In [191]:
import sqlite3

def pah_uid_report(uid):
    """
    Retrieve the transcription and transliteration for a given UID from the words_pahlavi table,
    along with definitions from the definitions_pahlavi table, dehkhoda_dictionary, and steingass_dictionary.
    
    Args:
        uid (int): The unique ID corresponding to UID in words_pahlavi.
        
    Returns:
        str: A formatted report of the transcription, transliteration, and definitions.
    """
    # Establish a connection to the database
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    
    try:
        # SQL query to retrieve the transcription, transliteration, and New_Persian for the given UID
        query = """
        SELECT Transcription, Translit_Skj, Translit_Mac, New_Persian
        FROM words_pahlavi
        WHERE UID = ?;
        """
        
        # Execute the query with the provided UID
        cursor.execute(query, (uid,))
        result = cursor.fetchone()

        # Check if a result was found
        if result:
            transcription, translit_skj, translit_mac, new_persian = result
            # Use Translit_Skj if available, otherwise use Translit_Mac
            translit = translit_skj if translit_skj else translit_mac
            
            # Prepare the first line with a double line break
            report = f"**{uid}: {transcription} ({translit})**\n\n"

            # Possible duplicates section
            duplicate_query = """
            SELECT UID, Transcription
            FROM words_pahlavi
            WHERE Transcription = ? AND UID != ?;
            """
            cursor.execute(duplicate_query, (transcription, uid))
            duplicate_matches = cursor.fetchall()

            # Format the possible duplicates output with a double line break after
            if duplicate_matches:
                report += "Possible duplicates:\n"
                for duplicate_uid, duplicate_transcription in duplicate_matches:
                    report += f"    {duplicate_uid}: {duplicate_transcription}\n"
                report += "\n"  # Double line break after possible duplicates

            # Query to get definitions corresponding to the Word_ID
            definition_query = """
            SELECT Type, Definition
            FROM definitions_pahlavi
            WHERE Word_ID = ?;
            """
            cursor.execute(definition_query, (uid,))
            definitions = cursor.fetchall()

            # Prepare definitions output
            report += "Definitions:\n"
            if definitions:
                for type_value, definition in definitions:
                    report += f"    {type_value}: {definition}\n"  # Indentation for definitions
            else:
                report += "    None\n"  # If no definitions are found

            # Look for exact matches in the malandra_pahlavi_dictionary
            malandra_query = """
            SELECT Definition, Etymology
            FROM malandra_pahlavi_dictionary
            WHERE transcription_skj = ?;
            """
            cursor.execute(malandra_query, (transcription,))
            malandra_results = cursor.fetchall()

            # Add Malandra definitions to the report
            if malandra_results:
                report += "\n\nMalandra definitions:\n"
                for malandra_definition, etymology in malandra_results:
                    report += f"    {malandra_definition}\n"  # Indentation for Malandra definitions

                # Check for etymology and add Cognates section if applicable
                if etymology:
                    report += "\nCognates:\n"
                    report += f"    {etymology}\n"  # Indentation for Cognates

            # Check for New Persian entry
            if new_persian:
                report += "\nNew Persian: " + new_persian + "\n"

                # Query the dehkhoda_dictionary for definitions based on New Persian
                dehkhoda_query = """
                SELECT Definition
                FROM dehkhoda_dictionary
                WHERE Word = ?;
                """
                cursor.execute(dehkhoda_query, (new_persian,))
                dehkhoda_definitions = cursor.fetchall()

                # Add Dehkhoda definitions to the report
                if dehkhoda_definitions:
                    report += "    Dehkhoda:\n"
                    for dehkhoda_definition in dehkhoda_definitions:
                        report += f"        {dehkhoda_definition[0]}\n"  # Further indentation for Dehkhoda definitions
                else:
                    report += "    Dehkhoda: None\n"

                # Query the steingass_dictionary for definitions based on New Persian
                steingass_query = """
                SELECT definitions
                FROM steingass_dictionary
                WHERE headword_persian = ?;
                """
                cursor.execute(steingass_query, (new_persian,))
                steingass_definitions = cursor.fetchall()

                # Add Steingass definitions to the report
                if steingass_definitions:
                    report += "    Steingass:\n"
                    for steingass_definition in steingass_definitions:
                        report += f"        {steingass_definition[0]}\n"  # Further indentation for Steingass definitions

            # Query the examples_pahlavi table for examples based on Word_ID
            examples_query = """
            SELECT Example, Translation
            FROM examples_pahlavi
            WHERE Word_ID = ?;
            """
            cursor.execute(examples_query, (uid,))
            examples = cursor.fetchall()

            # Add examples to the report
            if examples:
                report += "\n\nExamples:\n"
                for example, translation in examples:
                    report += f"    {example}: {translation}\n"  # Indentation for examples

            return report
        else:
            return f"No entry found for UID: {uid}"
    
    except Exception as e:
        return f"An error occurred: {e}"
    
    finally:
        # Close the cursor and connection
        cursor.close()
        conn.close()

# Example usage
# print(pah_uid_report(1))  # Replace 1 with the actual UID you want to query

In [193]:
print(pah_uid_report(786)) 

**786: bayaspān ([byʾspʾn' = M])**

Definitions:
    definition: messenger, envoy.


Malandra definitions:
    Prth by>sp>n; acc. to Nyb MP II 46 < OIr *dwaya-aspAna- 'having a two-horsed carriage'. But MacK has .


Examples:
    ﻿pas harw mardōm az bayaspān azd mad: ﻿"then everyone was informed by the envoy"

